In [1]:
from phm_america_2024.common.path_service_common import find_project_root
import duckdb
con = duckdb.connect()
import duckdb
import pandas as pd

import matplotlib.pyplot as plt

from IPython.display import display

In [2]:
# 1. Localiza la raíz del proyecto automáticamente
root = find_project_root()

# feature x train
x_train_path = root / "data/raw/train/X_train.csv"
# target y train
y_train_path = root / "data/raw/train/Y_train.csv"

# feature x test
x_test_path = root / "data/raw/test/X_test.csv"

# feature x validation
x_validation_path = root / "data/raw/validation/X_validation.csv"



In [4]:
# Definimos la query en DuckDB (une los CSVs y calcula las métricas en una sola pasada)
query_faulty = f"""
SELECT
    x.id,
    SUM(y.faulty) AS sum_faulty,
    COUNT(y.faulty) AS count_faulty,
    AVG(y.faulty) AS ratio_faulty
FROM read_csv_auto('{x_train_path}') x
JOIN read_csv_auto('{y_train_path}') y
  ON x.id = y.id
GROUP BY x.id
"""

# Ejecutamos la consulta y la convertimos a DataFrame
df_faulty_by_asset = con.sql(query_faulty).df()

# Mostramos el describe() tal como lo tenías antes
print(df_faulty_by_asset[['sum_faulty', 'count_faulty', 'ratio_faulty']].describe())

          sum_faulty  count_faulty   ratio_faulty
count  742625.000000      742625.0  742625.000000
mean        0.403189           1.0       0.403189
std         0.490538           0.0       0.490538
min         0.000000           1.0       0.000000
25%         0.000000           1.0       0.000000
50%         0.000000           1.0       0.000000
75%         1.000000           1.0       1.000000
max         1.000000           1.0       1.000000


In [19]:

# 1) Read CSVs (with consistent parameters)

X = pd.read_csv(x_train_path, sep=",", encoding="utf-8", decimal=".", low_memory=False)
Y = pd.read_csv(y_train_path, sep=",", encoding="utf-8", decimal=".", low_memory=False)
x_test_df = pd.read_csv(x_test_path, sep=",", encoding="utf-8", decimal=".",
                       low_memory=False)
x_validation_df = pd.read_csv(x_validation_path, sep=",", encoding="utf-8", decimal=""
                                                                          ".", low_memory=False)


In [20]:

# 2) The id column should be int for merging, ensure correct type
X["id"] = X["id"].astype(int)
Y["id"] = Y["id"].astype(int)

# 3) Consistency checks: check for duplicate ids in both X and Y
print("X dup id:", X["id"].duplicated().sum())
print("Y dup id:", Y["id"].duplicated().sum())

# 4) Merge on "id" with inner join and validate one-to-one relationship
df_train = X.merge(Y, on="id", how="inner", validate="one_to_one")

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("Merged shape:", df_train.shape)

# 5) Display first 10 rows of the merged DataFrame
display(df_train.head(10))


X dup id: 0
Y dup id: 0
X shape: (742625, 8)
Y shape: (742625, 3)
Merged shape: (742625, 10)


,id,trq_measured,oat,mgt,pa,ias,np,ng,faulty,trq_margin
0,0,54.10000,2.000000,544.5000,212.1408,74.562500,89.18000,99.64000,1,-13.717745
1,1,49.62500,24.222310,578.4844,1625.6400,30.355960,99.55273,91.38660,0,1.791863
2,2,52.00000,7.000000,566.1000,1912.9250,65.625000,100.14000,90.96000,1,-13.944871
3,3,62.40000,7.250000,560.1000,277.0632,54.812500,90.64000,100.28000,0,-0.017281
4,4,62.90000,23.250000,593.7000,53.6448,73.437500,99.91000,92.17000,0,7.322404
5,5,53.99805,16.512840,576.0781,1897.5110,47.871090,99.72461,92.32215,0,-1.004909
6,6,82.42969,24.203760,705.4063,1741.6400,28.614750,99.61915,99.45105,0,-6.174477
7,7,59.60000,11.500000,548.0000,-59.1312,76.875000,90.18000,99.95000,0,8.181553
8,8,67.61523,4.228794,628.3125,2140.9290,9.930908,99.65430,95.02332,1,-17.399932
9,9,81.90000,2.750000,659.5000,351.1296,109.500000,96.22000,99.91000,1,-15.639748


In [21]:

print("🔍 EVALUANDO ESTRATEGIA DE VALIDACIÓN CRUZADA (FASE 5)...\n")

query_cv_strategy = """
    SELECT
        CASE
            WHEN COUNT(*) = COUNT(DISTINCT id) THEN
                'ESCENARIO_A: Cada fila es independiente → usar StratifiedKFold'
            ELSE
                'ESCENARIO_B: Múltiples muestras por ID → usar GroupKFold por id'
        END as conclusion,
        COUNT(*) as total_rows,
        COUNT(DISTINCT id) as unique_ids,
        ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT id), 2) as avg_samples_per_id
    FROM df_train;
"""

# Ejecutar y mostrar el resultado
res_cv = con.execute(query_cv_strategy).df()
display(res_cv)

🔍 EVALUANDO ESTRATEGIA DE VALIDACIÓN CRUZADA (FASE 5)...



,conclusion,total_rows,unique_ids,avg_samples_per_id
0,ESCENARIO_A: Cada fila es independiente → usar...,742625,742625,1.0


In [22]:

# Read CSVs
df_x_train = pd.read_csv(x_train_path)
df_y_train = pd.read_csv(y_train_path)
df_x_test  = pd.read_csv(x_test_path)
df_x_validation   = pd.read_csv(x_validation_path)

In [23]:

con.register('X_df', df_x_train)
con.register('y_df', df_y_train)

print("🔍 ANÁLISIS AVANZADO: NATURALEZA Y PATRONES DEL 'ID'...\n")

# ---------------------------------------------------------
# QUERY 1: Estructura básica del ID
# ---------------------------------------------------------
print("1. Evaluando densidad y estructura básica de los IDs:")
query_1 = """
    SELECT
        'Dataset Train' as dataset,
        COUNT(*) as total_rows,
        COUNT(DISTINCT id) as unique_ids,
        ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT id), 2) as avg_samples_per_id,
        MIN(samples_per_id) as min_samples,
        MAX(samples_per_id) as max_samples,
        CASE
            WHEN COUNT(*) = COUNT(DISTINCT id) THEN 'INDEPENDIENTE'
            ELSE 'AGRUPADO_POR_ID'
        END as structure_type
    FROM (
        SELECT id, COUNT(*) as samples_per_id
        FROM X_df
        GROUP BY id
    );
"""
display(con.execute(query_1).df())

# ---------------------------------------------------------
# QUERY 2: Consistencia de labels por ID
# ---------------------------------------------------------
print("\n2. Buscando inconsistencias térmicas/fallos en el mismo ID:")
query_2 = """
    SELECT
        x.id,
        COUNT(*) as n_samples,
        COUNT(DISTINCT y.faulty) as unique_faulty_values,
        COUNT(DISTINCT y.trq_margin) as unique_margin_values,
        CASE
            WHEN COUNT(DISTINCT y.faulty) > 1 THEN 'INCONSISTENT_LABEL'
            ELSE 'CONSISTENT'
        END as label_consistency
    FROM X_df x
    JOIN y_df y ON x.id = y.id
    GROUP BY x.id
    HAVING COUNT(*) > 1 AND COUNT(DISTINCT y.faulty) > 1
    LIMIT 10;
"""
try:
    res_2 = con.execute(query_2).df()
    if len(res_2) == 0:
        print("✅ Resultado: Ningún ID presenta etiquetas inconsistentes (si hay repetidos, mantienen su estado).")
    else:
        display(res_2)
except Exception as e:
    print("Error en query 2:", e)

# ---------------------------------------------------------
# QUERY 3: Patrón numérico del ID
# ---------------------------------------------------------
print("\n3. Analizando la topología matemática de la secuencia de IDs:")
query_3 = """
    SELECT
        MIN(id) as min_id,
        MAX(id) as max_id,
        MAX(id) - MIN(id) as id_range,
        COUNT(DISTINCT id) as unique_ids,
        CASE
            WHEN MAX(id) - MIN(id) < COUNT(DISTINCT id) * 2 THEN 'SECUENCIAL_SIN_HUECOS'
            ELSE 'CON_HUECOS_O_PATRON'
        END as id_pattern
    FROM X_df;
"""
display(con.execute(query_3).df())

# ---------------------------------------------------------
# QUERY 4: Distribución de faulty por rango de IDs
# ---------------------------------------------------------
print("\n4. Buscando sesgos de lote (Batch Effects) por rango de IDs:")
query_4 = """
    SELECT
        CASE
            WHEN x.id <= 250000 THEN 'id_0_250k'
            WHEN x.id <= 500000 THEN 'id_250k_500k'
            WHEN x.id <= 750000 THEN 'id_500k_750k'
            ELSE 'id_750k_plus'
        END as id_range,
        COUNT(*) as n_samples,
        ROUND(AVG(y.faulty), 4) as faulty_ratio,
        COUNT(DISTINCT x.id) as unique_ids
    FROM X_df x
    JOIN y_df y ON x.id = y.id
    GROUP BY id_range
    ORDER BY id_range;
"""
display(con.execute(query_4).df())

🔍 ANÁLISIS AVANZADO: NATURALEZA Y PATRONES DEL 'ID'...

1. Evaluando densidad y estructura básica de los IDs:


,dataset,total_rows,unique_ids,avg_samples_per_id,min_samples,max_samples,structure_type
0,Dataset Train,742625,742625,1.0,1,1,INDEPENDIENTE



2. Buscando inconsistencias térmicas/fallos en el mismo ID:
✅ Resultado: Ningún ID presenta etiquetas inconsistentes (si hay repetidos, mantienen su estado).

3. Analizando la topología matemática de la secuencia de IDs:


,min_id,max_id,id_range,unique_ids,id_pattern
0,0,742624,742624,742625,SECUENCIAL_SIN_HUECOS



4. Buscando sesgos de lote (Batch Effects) por rango de IDs:


,id_range,n_samples,faulty_ratio,unique_ids
0,id_0_250k,250001,0.4028,250001
1,id_250k_500k,250000,0.4032,250000
2,id_500k_750k,242624,0.4035,242624


In [24]:

con.register('X_train_df', df_x_train)
con.register('X_val_df', df_x_validation)
con.register('X_test_df', df_x_test)

print("🔍 ANALIZANDO DOMAIN SHIFT (DRIFT) ENTRE SETS...\n")

query_drift = """
    WITH combined AS (
        SELECT 'train' as split, trq_measured, oat, mgt, pa, ias, np, ng FROM X_train_df
        UNION ALL
        SELECT 'validation' as split, trq_measured, oat, mgt, pa, ias, np, ng FROM X_val_df
        UNION ALL
        SELECT 'test' as split, trq_measured, oat, mgt, pa, ias, np, ng FROM X_test_df
    )
    SELECT
        split,
        COUNT(*) as n_samples,
        ROUND(AVG(trq_measured), 2) as avg_trq,
        ROUND(STDDEV(trq_measured), 2) as std_trq,
        ROUND(AVG(oat), 2) as avg_oat,
        ROUND(AVG(mgt), 2) as avg_mgt,
        ROUND(AVG(pa), 2) as avg_pa
    FROM combined
    GROUP BY split
    ORDER BY split;
"""

display(con.execute(query_drift).df())

🔍 ANALIZANDO DOMAIN SHIFT (DRIFT) ENTRE SETS...



,split,n_samples,avg_trq,std_trq,avg_oat,avg_mgt,avg_pa
0,test,21436,71.85,12.30,14.53,604.36,320.64
1,train,742625,65.10,13.25,12.68,592.25,511.78
2,validation,21436,63.89,12.55,14.99,582.83,575.19


In [25]:

con.register('X_df', df_x_train)
con.register('y_df', df_y_train)

print("🔍 INICIANDO ANÁLISIS ESTRUCTURAL DE IDs...\n")

# ---------------------------------------------------------
# PREGUNTA 1: ¿El "id" tiene un patrón que identifique el activo?
# ---------------------------------------------------------
print("1. Buscando patrones de agrupación en los prefijos de los IDs...")
query_1 = """
    SELECT
        id,
        -- Extraer prefijo forzando la división a entero (CAST)
        CASE
            WHEN id < 100000 THEN CAST(id / 1000 AS INTEGER)
            WHEN id < 1000000 THEN CAST(id / 10000 AS INTEGER)
            ELSE CAST(id / 100000 AS INTEGER)
        END as potential_asset_group,
        COUNT(*) as samples_per_id
    FROM X_df
    GROUP BY id
    ORDER BY id
    LIMIT 20;
"""
display(con.execute(query_1).df())

# ---------------------------------------------------------
# PREGUNTA 2: ¿Hay IDs repetidos (mismo activo, múltiples mediciones)?
# ---------------------------------------------------------
print("\n2. Verificando si existen mediciones repetidas para un mismo ID...")
query_2 = """
    SELECT
        x.id,
        COUNT(*) as n_samples,
        COUNT(DISTINCT y.faulty) as n_different_labels,
        COUNT(DISTINCT y.trq_margin) as n_different_margins
    FROM X_df x
    JOIN y_df y ON x.id = y.id
    GROUP BY x.id
    HAVING COUNT(*) > 1
    ORDER BY n_samples DESC
    LIMIT 20;
"""
# Usamos un try-except por si la consulta viene vacía (lo cual es bueno si no hay repetidos)
try:
    res_2 = con.execute(query_2).df()
    if len(res_2) == 0:
        print("✅ Resultado: No hay IDs repetidos en el dataset.")
    else:
        display(res_2)
except Exception as e:
    print("Error en query 2:", e)

# ---------------------------------------------------------
# PREGUNTA 3: Buscar columnas ocultas o metadatos
# ---------------------------------------------------------
print("\n3. Describiendo la estructura y tipos de datos del dataset...")
query_3 = """
    DESCRIBE X_df;
"""
display(con.execute(query_3).df())

# ---------------------------------------------------------
# PREGUNTA 4: ¿Existe alguna columna que agrupe muestras?
# ---------------------------------------------------------
print("\n4. Diagnóstico final: ¿Son series de tiempo agrupadas o filas independientes?")
query_4 = """
    SELECT
        COUNT(DISTINCT id) as unique_ids,
        COUNT(*) as total_rows,
        CASE
            WHEN COUNT(DISTINCT id) = COUNT(*) THEN 'CADA_FILA_ES_INDEPENDIENTE'
            ELSE 'HAY_AGRUPACION_POR_ID'
        END as data_structure
    FROM X_df;
"""
display(con.execute(query_4).df())

🔍 INICIANDO ANÁLISIS ESTRUCTURAL DE IDs...

1. Buscando patrones de agrupación en los prefijos de los IDs...


,id,potential_asset_group,samples_per_id
0,0,0,1
1,1,0,1
2,2,0,1
3,3,0,1
4,4,0,1
5,5,0,1
6,6,0,1
7,7,0,1
8,8,0,1
9,9,0,1



2. Verificando si existen mediciones repetidas para un mismo ID...
✅ Resultado: No hay IDs repetidos en el dataset.

3. Describiendo la estructura y tipos de datos del dataset...


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,trq_measured,DOUBLE,YES,None,None,None
2,oat,DOUBLE,YES,None,None,None
3,mgt,DOUBLE,YES,None,None,None
4,pa,DOUBLE,YES,None,None,None
5,ias,DOUBLE,YES,None,None,None
6,np,DOUBLE,YES,None,None,None
7,ng,DOUBLE,YES,None,None,None



4. Diagnóstico final: ¿Son series de tiempo agrupadas o filas independientes?


,unique_ids,total_rows,data_structure
0,742625,742625,CADA_FILA_ES_INDEPENDIENTE


In [27]:
con.register('df_x_train', df_x_train)
con.register('df_y_train', df_y_train)
con.register('df_x_validation', df_x_validation)
con.register('df_x_test', df_x_test)

query_sizes = """
SELECT 'df_x_train'    AS dataset, COUNT(*) AS rows FROM df_x_train
UNION ALL
SELECT 'df_y_train'    AS dataset, COUNT(*) AS rows FROM df_y_train
UNION ALL
SELECT 'df_x_validation' AS dataset, COUNT(*) AS rows FROM df_x_validation
UNION ALL
SELECT 'df_x_test'       AS dataset, COUNT(*) AS rows FROM df_x_test;
"""

display(con.execute(query_sizes).df())

,dataset,rows
0,df_x_train,742625
1,df_y_train,742625
2,df_x_validation,21436
3,df_x_test,21436


In [29]:
for name in ["df_x_train", "df_y_train", "df_x_validation", "df_x_test"]:
    print(f"\n--- DESCRIBE {name} ---")
    display(con.execute(f"DESCRIBE {name}").df())


--- DESCRIBE df_x_train ---


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,trq_measured,DOUBLE,YES,None,None,None
2,oat,DOUBLE,YES,None,None,None
3,mgt,DOUBLE,YES,None,None,None
4,pa,DOUBLE,YES,None,None,None
5,ias,DOUBLE,YES,None,None,None
6,np,DOUBLE,YES,None,None,None
7,ng,DOUBLE,YES,None,None,None



--- DESCRIBE df_y_train ---


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,faulty,BIGINT,YES,None,None,None
2,trq_margin,DOUBLE,YES,None,None,None



--- DESCRIBE df_x_validation ---


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,trq_measured,DOUBLE,YES,None,None,None
2,oat,DOUBLE,YES,None,None,None
3,mgt,DOUBLE,YES,None,None,None
4,pa,DOUBLE,YES,None,None,None
5,ias,DOUBLE,YES,None,None,None
6,np,DOUBLE,YES,None,None,None
7,ng,DOUBLE,YES,None,None,None



--- DESCRIBE df_x_test ---


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,trq_measured,DOUBLE,YES,None,None,None
2,oat,DOUBLE,YES,None,None,None
3,mgt,DOUBLE,YES,None,None,None
4,pa,DOUBLE,YES,None,None,None
5,ias,DOUBLE,YES,None,None,None
6,np,DOUBLE,YES,None,None,None
7,ng,DOUBLE,YES,None,None,None


In [30]:
# Read CSVs
df_x_train = pd.read_csv(x_train_path)
df_y_train = pd.read_csv(y_train_path)
df_x_test  = pd.read_csv(x_test_path)
df_x_validation   = pd.read_csv(x_validation_path)

con.register('X', df_x_train)
con.register('Y', df_y_train)

In [31]:
# Use SQL to join X_train and Y_train...
print("Merging X_train and Y_train...")
# Use SQL to join the tables on the 'id' column
query_unificada = """
    SELECT x.*, y.faulty, y.trq_margin
    FROM X as x
    JOIN y as y ON x.id = y.id
"""
# Save the result in a new master DataFrame
df_train = con.execute(query_unificada).df()
print("Dataset successfully merged. Shape:", df_train.shape)

# ---------------------------------------------------------
# 2. PHYSICAL EXPLORATORY QUESTIONS (Data Understanding)
# ---------------------------------------------------------

# Question A: What is the class imbalance? (How many healthy vs failures?)
query_clases = """
    SELECT
        faulty as estado_motor,
        COUNT(*) as cantidad,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM df_train), 2) as porcentaje
    FROM df_train
    GROUP BY faulty;
"""
print("\n--- CLASS DISTRIBUTION ---")
display(con.execute(query_clases).df())

# Question B: Does gas temperature (mgt) rise when there is a failure (faulty=1)?
query_temperatura = """
    SELECT
        faulty as estado_motor,
        ROUND(AVG(mgt), 2) as temp_gas_media,
        ROUND(AVG(oat), 2) as temp_exterior_media,
        ROUND(AVG(trq_measured), 2) as torque_medido_medio
    FROM df_train
    GROUP BY faulty;
"""
print("\n--- PHYSICAL SYMPTOMS ANALYSIS (TEMPERATURE AND TORQUE) ---")
display(con.execute(query_temperatura).df())

Merging X_train and Y_train...
Dataset successfully merged. Shape: (742625, 10)

--- CLASS DISTRIBUTION ---


,estado_motor,cantidad,porcentaje
0,1,299418,40.32
1,0,443207,59.68



--- PHYSICAL SYMPTOMS ANALYSIS (TEMPERATURE AND TORQUE) ---


,estado_motor,temp_gas_media,temp_exterior_media,torque_medido_medio
0,0,586.24,14.92,66.54
1,1,601.14,9.35,62.96


In [32]:
# Assuming you already have your unified dataframe 'df_train' (X + y)
# If not, make sure you ran the JOIN first

# 1. Physical symptoms analysis: Comparison of means and dispersion (Uncertainty)
query_fisica = """
    SELECT
        faulty as estado_motor,
        -- Thermal and power variables
        ROUND(AVG(mgt), 2) as avg_mgt,
        ROUND(STDDEV(mgt), 2) as std_mgt,
        ROUND(AVG(trq_measured), 2) as avg_trq,
        ROUND(STDDEV(trq_measured), 2) as std_trq,
        -- Environmental variable
        ROUND(AVG(oat), 2) as avg_oat,
        COUNT(*) as n_samples
    FROM df_train
    GROUP BY faulty
    ORDER BY faulty;
"""

# 2. Correlation analysis: Which variable most "drives" the failure?
query_correlacion = """
    SELECT
        CORR(faulty, mgt) as corr_mgt_faulty,
        CORR(faulty, trq_measured) as corr_trq_faulty,
        CORR(faulty, pa) as corr_pa_faulty,
        CORR(faulty, ng) as corr_ng_faulty
    FROM df_train;
"""

print("--- ANÁLISIS DE SÍNTOMAS FÍSICOS (MEDIA Y DISPERSIÓN) ---")
df_fisica = duckdb.query(query_fisica).df()
display(df_fisica)

print("\n--- CORRELACIÓN DE VARIABLES CON EL ESTADO DE FALLO ---")
df_corr = duckdb.query(query_correlacion).df()
display(df_corr)

# Quick visual interpretation
print("\nNota: Una correlación positiva en 'mgt' indica que a mayor temperatura, mayor probabilidad de fallo.")

--- ANÁLISIS DE SÍNTOMAS FÍSICOS (MEDIA Y DISPERSIÓN) ---


,estado_motor,avg_mgt,std_mgt,avg_trq,std_trq,avg_oat,n_samples
0,0,586.24,38.89,66.54,11.21,14.92,443207
1,1,601.14,39.10,62.96,15.56,9.35,299418



--- CORRELACIÓN DE VARIABLES CON EL ESTADO DE FALLO ---


,corr_mgt_faulty,corr_trq_faulty,corr_pa_faulty,corr_ng_faulty
0,0.184274,-0.132677,0.342304,-0.090337



Nota: Una correlación positiva en 'mgt' indica que a mayor temperatura, mayor probabilidad de fallo.


In [ ]:
# ###############################################################################

In [33]:

# 1. Merge datasets: Joining sensor features (X) with target labels (Y)
print("Merging X_train and Y_train...")
query_merged = """
    SELECT x.*, y.faulty, y.trq_margin
    FROM X as x
    JOIN y as y ON x.id = y.id
"""
df_train = con.execute(query_merged).df()
print(f"Dataset successfully merged. Shape: {df_train.shape}")


Merging X_train and Y_train...
Dataset successfully merged. Shape: (742625, 10)


In [34]:

# ---------------------------------------------------------
# 2. DATA UNDERSTANDING: PHYSICS-BASED PROFILING
# ---------------------------------------------------------

# Query 1: Physical Limits Assessment
# Purpose: Check for zero/negative values in OAT and NG to prevent division-by-zero
# errors in KPI engineering (Fase 3).
query_limits = """
    SELECT
        MIN(oat) as min_oat, MAX(oat) as max_oat,
        MIN(ng) as min_ng, MAX(ng) as max_ng,
        COUNT(*) as total_samples
    FROM df_train;
"""
print("\n--- CHECKING PHYSICAL LIMITS (PHASE 3 CONFIG) ---")
display(con.execute(query_limits).df())



--- CHECKING PHYSICAL LIMITS (PHASE 3 CONFIG) ---


,min_oat,max_oat,min_ng,max_ng,total_samples
0,-19.25,36.79913,90.00183,101.22,742625


In [35]:

# Query 2: Target Distribution Analysis
# Purpose: Calculate skewness of the target to choose the correct NGBoost
# distribution (Normal vs LogNormal) in Fase 4.
query_distribution = """
    SELECT
        AVG(trq_margin) as mu,
        STDDEV(trq_margin) as sigma,
        (AVG(POWER(trq_margin - (SELECT AVG(trq_margin) FROM df_train), 3)) /
         POWER(STDDEV(trq_margin), 3)) as skewness
    FROM df_train;
"""
print("\n--- TARGET DISTRIBUTION & SKEWNESS (PHASE 4 CONFIG) ---")
display(con.execute(query_distribution).df())



--- TARGET DISTRIBUTION & SKEWNESS (PHASE 4 CONFIG) ---


,mu,sigma,skewness
0,-1.157346,14.068843,-2.752006


In [36]:

# Query 3: Multi-collinearity Assessment
# Purpose: Identify highly correlated sensor inputs to prune redundant features
# and avoid overfitting in LightGBM/NGBoost.
query_collinearity = """
    SELECT
        CORR(pa, trq_measured) as corr_pa_trq,
        CORR(mgt, trq_measured) as corr_mgt_trq,
        CORR(ng, np) as corr_ng_np
    FROM df_train;
"""
print("\n--- FEATURE REDUNDANCY CHECK (PHASE 4 CONFIG) ---")
display(con.execute(query_collinearity).df())



--- FEATURE REDUNDANCY CHECK (PHASE 4 CONFIG) ---


,corr_pa_trq,corr_mgt_trq,corr_ng_np
0,-0.179484,0.643654,-0.789165


In [37]:

# Query 4: Operation Regime Profiling
# Purpose: Verify if 'n_components=4' is appropriate for GMM-based
# Group-K-Fold validation (Fase 5).
query_regimes = """
    SELECT
        ROUND(oat, -1) as oat_bin,
        COUNT(*) as density
    FROM df_train
    GROUP BY oat_bin
    ORDER BY density DESC
    LIMIT 10;
"""
print("\n--- FLIGHT REGIME PROFILING (PHASE 5 CONFIG) ---")
display(con.execute(query_regimes).df())


--- FLIGHT REGIME PROFILING (PHASE 5 CONFIG) ---


,oat_bin,density
0,20.0,297713
1,10.0,295229
2,0.0,107971
3,30.0,21017
4,-10.0,12181
5,-20.0,8459
6,40.0,55


In [38]:
query_final_verification = """
SELECT
    -- Leakage check (PENDING crítico)
    CORR(trq_measured, trq_margin)      AS corr_trq_leakage,

    -- Rangos físicos
    MIN(oat)                            AS min_oat,
    MAX(oat)                            AS max_oat,
    MIN(ng)                             AS min_ng,
    MAX(ng)                             AS max_ng,

    -- Target de regresión
    AVG(trq_margin)                     AS mu_trq_margin,
    STDDEV(trq_margin)                  AS sigma_trq_margin,

    -- Tamaño real
    COUNT(*)                            AS total_rows

FROM df_train;
"""

print("\n--- PHASE 2 FINAL VERIFICATION ---")
display(con.execute(query_final_verification).df())


--- PHASE 2 FINAL VERIFICATION ---


,corr_trq_leakage,min_oat,max_oat,min_ng,max_ng,mu_trq_margin,sigma_trq_margin,total_rows
0,0.338792,-19.25,36.79913,90.00183,101.22,-1.157346,14.068843,742625


In [39]:
print(con.execute("DESCRIBE df_train").df())

    column_name column_type null   key default extra
0            id      BIGINT  YES  None    None  None
1  trq_measured      DOUBLE  YES  None    None  None
2           oat      DOUBLE  YES  None    None  None
3           mgt      DOUBLE  YES  None    None  None
4            pa      DOUBLE  YES  None    None  None
5           ias      DOUBLE  YES  None    None  None
6            np      DOUBLE  YES  None    None  None
7            ng      DOUBLE  YES  None    None  None
8        faulty      BIGINT  YES  None    None  None
9    trq_margin      DOUBLE  YES  None    None  None
